In [12]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.layers import Dense , Embedding , Lambda 
import random
from sklearn.model_selection import train_test_split
import re

In [13]:
# Data Collection :
df = pd.read_csv("/Users/aayushagarwal/Downloads/LP IV Updated Datasets/CBOW/IMDB Dataset.csv")
print(df.shape)

labels = (df["sentiment"] == "positive").astype(int)
train_texts , test_texts , y_train , y_test = train_test_split(df["review"].tolist() , labels.values , test_size=0.2 , random_state=1)
print(len(train_texts) , len(test_texts))


(50000, 2)
40000 10000


In [14]:
# Tokenization :
tokenizer = Tokenizer(num_words=5000)   # keep top 5000 words (index 0 stays free for padding)
tokenizer.fit_on_texts(train_texts)
word2idx = tokenizer.word_index
idx2word = tokenizer.index_word
vocab_size = 5000
train_seqs = tokenizer.texts_to_sequences(train_texts)
test_seqs = tokenizer.texts_to_sequences(test_texts)
print("Vocab Size : " , vocab_size)

Vocab Size :  5000


In [15]:
# Generate CBOW Training Data : (use only 3000 reviews to keep it fast)
window_size = 2
contexts , targets = [] , []

for seq in train_seqs[:3000]:
    for i in range(window_size , len(seq) - window_size):
        contexts.append(seq[i - window_size : i] + seq[i + 1 : i + window_size + 1])
        targets.append(seq[i])

contexts = np.array(contexts)
targets = np.array(targets)
print("Training samples : " , len(contexts))

Training samples :  623057


In [16]:
# CBOW Model :
cbow = keras.Sequential([
    Embedding(vocab_size , 100),
    Lambda(lambda x : tf.reduce_mean(x , axis=1)),
    Dense(vocab_size , activation="softmax")
])

cbow.compile(optimizer="adam" , loss="sparse_categorical_crossentropy" , metrics=["accuracy"])
cbow.fit(contexts , targets , epochs=5 , batch_size=1024)

Epoch 1/5
609/609 ━━━━━━━━━━━━━━━━━━━━ 15s 23ms/step - accuracy: 0.0819 - loss: 6.5715
Epoch 2/5
609/609 ━━━━━━━━━━━━━━━━━━━━ 13s 21ms/step - accuracy: 0.1054 - loss: 5.9632
Epoch 3/5
609/609 ━━━━━━━━━━━━━━━━━━━━ 13s 21ms/step - accuracy: 0.1258 - loss: 5.7605
Epoch 4/5
609/609 ━━━━━━━━━━━━━━━━━━━━ 13s 21ms/step - accuracy: 0.1390 - loss: 5.6221
Epoch 5/5
609/609 ━━━━━━━━━━━━━━━━━━━━ 15s 25ms/step - accuracy: 0.1474 - loss: 5.5159


In [17]:
# Word Embeddings : (weights of the trained CBOW Embedding layer)
embeddings = cbow.layers[0].get_weights()[0]
print(embeddings.shape)

(5000, 100)


In [18]:
# Review Vectors : average of the embeddings of all words in the review
def review_vector(seq):
    seq = [w for w in seq if w < vocab_size]
    if len(seq) == 0:
        return np.zeros(embeddings.shape[1])
    return embeddings[seq].mean(axis=0)

X_train = np.array([review_vector(s) for s in train_seqs])
X_test = np.array([review_vector(s) for s in test_seqs])
print(X_train.shape , X_test.shape)

(40000, 100) (10000, 100)


In [19]:
# Sentiment Classifier : (on top of CBOW embeddings)
model = keras.Sequential([
    Dense(32 , activation="relu"),
    Dense(1 , activation="sigmoid")
])

model.compile(optimizer="adam" , loss="binary_crossentropy" , metrics=["accuracy"])

In [20]:
# Train :
model.fit(X_train , y_train , epochs=10 , batch_size=64 , validation_split=0.1)

Epoch 1/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 1s 504us/step - accuracy: 0.6101 - loss: 0.6613 - val_accuracy: 0.6457 - val_loss: 0.6419
Epoch 2/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 0s 385us/step - accuracy: 0.6449 - loss: 0.6335 - val_accuracy: 0.6685 - val_loss: 0.6190
Epoch 3/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 0s 382us/step - accuracy: 0.6629 - loss: 0.6169 - val_accuracy: 0.6690 - val_loss: 0.6090
Epoch 4/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 0s 378us/step - accuracy: 0.6717 - loss: 0.6082 - val_accuracy: 0.6920 - val_loss: 0.5973
Epoch 5/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 0s 380us/step - accuracy: 0.6769 - loss: 0.6020 - val_accuracy: 0.6910 - val_loss: 0.5925
Epoch 6/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 0s 380us/step - accuracy: 0.6821 - loss: 0.5967 - val_accuracy: 0.6812 - val_loss: 0.5952
Epoch 7/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 0s 379us/step - accuracy: 0.6852 - loss: 0.5916 - val_accuracy: 0.6915 - val_loss: 0.5869
Epoch 8/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 0s 402us/step - accuracy: 0.6873 - loss: 0.5892 - 

In [21]:
# Evaluate :
loss , acc = model.evaluate(X_test , y_test)
print("Test Loss : " , loss)
print("Test Accuracy : " , acc)

313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 297us/step - accuracy: 0.6871 - loss: 0.5896
Test Loss :  0.5895609259605408
Test Accuracy :  0.6870999932289124


In [22]:
# Prediction :
n = random.randint(0 , len(X_test) - 1)
pred = model.predict(X_test[n:n+1])[0][0]

print("Review : " , test_texts[n][:150] , "...")
print("Actual : " , "Positive" if y_test[n] == 1 else "Negative")
print("Predicted : " , "Positive" if pred > 0.5 else "Negative")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
Review :  this movie was so gay like its a mom and son cat that have sex, they also get scared of little kitty cats. they get set on fire by them. the mom cat a ...
Actual :  Negative
Predicted :  Positive
